# Ytools-Story — YouTube Faceless Automation

Loop footage + overlay (watermark, particles, audio spectrum, karaoke subtitle) + TTS narration.
Bawa script sendiri (wajib) atau pakai LLM (openai/anthropic, butuh API key). Bilingual ID/EN.

**Cara pakai:** Run semua cell top-to-bottom (Shift+Enter).
Upload footage di **Cell 4**, atur opsi di **Cell 5**, upload script di **Cell 6**, lalu lihat hasil di **Cell 7-8**.


In [ ]:
#@title 1. Install dependencies (~1 min)
!pip install -q edge-tts ffmpeg-python imageio-ffmpeg pillow numpy pyyaml

import os, sys, shutil, subprocess
print('python', sys.version.split()[0])

REPO = 'https://github.com/fawaz333888/Ytools-Story.git'
DEST = '/content/Ytools-Story'

def _have_ytools(path):
    return os.path.isfile(os.path.join(path, 'ytools', '__init__.py'))

if not _have_ytools(DEST):
    ok = subprocess.run(['git', 'clone', '-q', REPO, DEST]).returncode == 0
    if not ok or not _have_ytools(DEST):
        # fallback: copy from the cwd this notebook was launched from
        src = os.getcwd()
        if _have_ytools(src) and os.path.abspath(src) != os.path.abspath(DEST):
            shutil.copytree(src, DEST, dirs_exist_ok=True)
            print('clone failed - copied local copy from', src)
        else:
            print('ERROR: clone gagal dan tidak ada copy lokal. Push repo ke GitHub dulu.')
else:
    # already cloned: always pull latest so notebook changes take effect
    subprocess.run(['git', '-C', DEST, 'fetch', '-q'])
    subprocess.run(['git', '-C', DEST, 'reset', '-q', '--hard', 'origin/master'])
    print('updated to latest')

os.chdir(DEST)
sys.path.insert(0, DEST)

from ytools import __version__
print('ytools', __version__)

In [ ]:
#@title 2. Cek environment (GPU / ffmpeg)
import subprocess
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout or 'no GPU')
except Exception as e:
    print('nvidia-smi error:', e)

from ytools.render.ffutil import FFRunner
ff = FFRunner()
print('ffmpeg:', ff.version())
print('ffprobe:', ff.ffprobe or 'tidak ada (fallback parse ffmpeg -i)')
print('nvenc flag:', ff.supports_nvenc())

In [ ]:
#@title 3. Mount Google Drive (sekali, untuk simpan hasil)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Ytools-Story'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive siap:', DRIVE_ROOT)

In [ ]:
#@title 4. Upload footage video (minimal 30 detik)
from google.colab import files
import shutil, os

FOOTAGE = 'footage.mp4'
if not os.path.isfile(FOOTAGE):
    uploaded = files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        shutil.move(name, FOOTAGE)
        print('saved as', FOOTAGE)
    else:
        print('tidak ada file diupload')

if os.path.isfile(FOOTAGE):
    info = ff.probe(FOOTAGE)
    print(f"footage: {info['width']}x{info['height']} {info['duration']:.1f}s fps={info['fps']:.1f} audio={info['has_audio']}")
    if info['duration'] < 30:
        print('WARNING: footage < 30 detik. Loop terlalu repetitif untuk YouTube.')

In [ ]:
#@title 5. Konfigurasi
#@markdown --- **Story** ---
niche = 'horror' #@param ['horror','motivation','education','drama','custom']
#@markdown *Tema cerita. `custom` = cerita netral tanpa gaya khusus.*
language = 'id' #@param ['id','en']
#@markdown *Bahasa narasi + subtitle. Voice TTS auto-milih sesuai ini.*
provider = 'manual' #@param ['manual','openai','anthropic']
#@markdown *`manual` = bawa script sendiri (upload di cell 6). LLM = generate cerita otomatis (butuh API key di bawah).*
topic = '' #@param {type:'string'}
#@markdown *Hanya untuk LLM: tema spesifik cerita. Kosong = bebas.*
length_minutes = 3 #@param {type:'number'}
#@markdown *Hanya untuk LLM (perkiraan, ~130 kata/menit). `manual` mengikuti panjang script.*
seed = None #@param {type:'raw'}
#@markdown *Angka untuk hasil LLM yang reproducible. `None` = acak tiap run.*

#@markdown --- **Narration (TTS)** ---
voice = 'auto' #@param ['auto','id-ID-GadisNeural','id-ID-ArdiNeural','en-US-AvaNeural','en-US-AndrewNeural','en-US-EmmaNeural','en-US-BrianNeural','en-GB-SoniaNeural','en-GB-RyanNeural']
#@markdown *Suara narator. `auto` = sesuai language. Gadis/Ardi = Indonesia (cuma 2 yang ada).*
tts_rate = '+0%' #@param ['-20%','-10%','+0%','+10%','+20%']
#@markdown *Kecepatan bicara. `+10%` = sedikit lebih cepat (cocok YouTube).*

#@markdown --- **Video** ---
resolution = '1280x720' #@param ['854x480','1280x720','1920x1080']
#@markdown *Resolusi output. 720p = standar YouTube; 1080p = lebih tajam tapi render 2x lebih lama.*
fps = 30 #@param ['24','30','60']
#@markdown *Frame rate. 30 = standar; 60 = mulus tapi file lebih besar.*
motion = 'slow_drift' #@param ['none','slow_drift','slow_zoom']
#@markdown *Gerakan kamera virtual. `slow_drift` = geser perlahan; `slow_zoom` = zoom nafas; `none` = statis.*
motion_intensity = 1.0 #@param {type:'number'}
#@markdown *Kuat & cepat gerakan. 1.0 = default (pelan); 1.5-2.0 = lebih jelas; 0.5 = sangat halus. Range 0-3.*

#@markdown --- **Overlays** ---
particle_style = 'dust' #@param ['dust','snow','sparkle','fireflies','embers','fog']
#@markdown *Efek partikel di seluruh layar. `fog` = kabut, `embers` = bara api.*
particle_density = 120 #@param {type:'integer'}
#@markdown *Jumlah titik partikel. 60 = tipis, 240 = lebat.*
particle_size = 1.0 #@param {type:'number'}
#@markdown *Ukuran titik. 1.0 = default; 3.0 = 3x lebih besar.*
particle_opacity = 1.0 #@param {type:'number'}
#@markdown *Transparansi partikel. 1.0 = penuh; 0.5 = tipis; 1.5-2.0 = lebih tegas kalau masih samar.*
watermark_text = '@YtoolsChannel' #@param {type:'string'}
#@markdown *Nama channel — muncul sebagai badge kanan atas (channel tag).*
watermark_style = 'badge' #@param ['badge','plain','logo']
#@markdown *`badge` = kotak gelap; `plain` = teks polos; `logo` = upload gambar logo (cell 6).*
watermark_position = 'top-right' #@param ['top-left','top-right','bottom-left','bottom-right','center']
#@markdown *Posisi watermark. `top-right` = kanan atas (channel tag).*
watermark_opacity = 0.85 #@param {type:'number'}
#@markdown *Transparansi watermark. 1.0 = penuh; 0.5 = setengah pudar. Diterapkan saat compositing: gak perlu render ulang.*
subtitle_style = 'karaoke' #@param ['karaoke','simple']
#@markdown *`karaoke` = highlight kata per kata; `simple` = teks biasa.*

#@markdown --- **Audio spectrum** (visualizer narasi, default off) ---
spectrum_enabled = False #@param {type:'boolean'}
#@markdown *Visualizer gelombang audio dari suara narasi. Naikin render time di CPU.*
spectrum_style = 'cqt' #@param ['cqt','spectrum','waves','vectorscope']
#@markdown *`cqt` = bar graf (paling cocok untuk bicara); `spectrum` = waterfall warna; `waves` = gelombang garis; `vectorscope` = lissajous bulat.*
spectrum_position = 'bottom' #@param ['bottom','center']
#@markdown *Posisi band. `bottom` bisa nabrak subtitle: naikin margin_v atau pakai `center`.*
spectrum_height = 0 #@param {type:'integer'}
#@markdown *Ketebalan band dalam piksel. 0 = auto (20% tinggi video: 144px @720p). **80 = lebih tipis dari auto**; untuk band tebal isi 200-280.*
spectrum_opacity = 0.9 #@param {type:'number'}
#@markdown *Transparansi spectrum. 1.0 = penuh; 0.5 = setengah pudar.*
spectrum_palette = 'white' #@param ['white','green','amber','cyan']
#@markdown *Warna spectrum. `white` = bersih; `green` = klasik; `amber` = hangat; `cyan` = dingin.*

#@markdown --- **Card layout** (thumbnail kiri atas + judul; channel tag = watermark) ---
card_enabled = False #@param {type:'boolean'}
#@markdown *Thumbnail + judul di atas. Upload gambar thumbnail di cell 6.*
card_title = '' #@param {type:'string'}
#@markdown *Judul di samping thumbnail. Kosong + provider LLM = LLM generate judul; `manual` wajib isi.*

#@markdown --- **Subscribe button** (pill merah animasi, default off) ---
subscribe_enabled = False #@param {type:'boolean'}
#@markdown *Tombol subscribe di bawah watermark (kanan atas). Muncul dengan animasi bounce-in + goyangan pelan sampai akhir video.*
subscribe_text = 'SUBSCRIBE' #@param {type:'string'}
#@markdown *Teks tombol. Default `SUBSCRIBE`.*
subscribe_appear = 3.0 #@param {type:'number'}
#@markdown *Detik muncul (bounce-in). 3.0 = muncul di detik ke-3. Goyangan: 6px setiap 2.5 detik (fixed).*

#@markdown --- **Output** ---
output_name = 'video.mp4' #@param {type:'string'}
#@markdown *Nama file output (disimpan ke Drive di cell 8).*
video_bitrate = '6M' #@param ['3M','4M','6M','8M']
#@markdown *Target bitrate video. 6M = ~90MB/3menit; turunkan ke 4M/3M kalau file terlalu besar.*

#@markdown --- **LLM key (hanya jika provider != manual)** ---
llm_model = 'gpt-4o-mini' #@param {type:'string'}
#@markdown *Model LLM. Contoh: `gpt-4o-mini` (OpenAI), `claude-3-5-sonnet-20241022` (Anthropic).*
llm_base_url = '' #@param {type:'string'}
#@markdown *Kosong = endpoint resmi; isi untuk OpenAI-compatible (OpenRouter/Groq/vLLM).*
llm_api_key = '' #@param {type:'string'}
#@markdown *Kosong = pakai Colab secret; JANGAN isi kalau notebook dibagikan.*
os.environ.pop('OPENAI_API_KEY', None)
if provider == 'openai':
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = llm_api_key or userdata.get('OPENAI_API_KEY', '')
elif provider == 'anthropic':
    from google.colab import userdata
    os.environ.pop('ANTHROPIC_API_KEY', None)
    os.environ['ANTHROPIC_API_KEY'] = llm_api_key or userdata.get('ANTHROPIC_API_KEY', '')

from ytools.config import Config
cfg = Config()
cfg.set('story.niche', niche); cfg.set('story.language', language)
cfg.set('story.provider', provider); cfg.set('story.topic', topic)
cfg.set('story.length_minutes', length_minutes)
cfg.set('story.model', llm_model or None)
cfg.set('story.base_url', llm_base_url or None)
if provider == 'anthropic':
    cfg.set('story.api_key_env', 'ANTHROPIC_API_KEY')
cfg.set('story.seed', seed)
cfg.set('tts.voice', None if voice == 'auto' else voice); cfg.set('tts.rate', tts_rate)
_res_w, _res_h = resolution.split('x')
cfg.set('video.width', int(_res_w)); cfg.set('video.height', int(_res_h))
cfg.set('video.fps', int(fps)); cfg.set('video.motion', motion)
cfg.set('video.motion_intensity', motion_intensity)
cfg.set('overlays.particles.style', particle_style)
cfg.set('overlays.particles.density', particle_density)
cfg.set('overlays.particles.size', particle_size)
cfg.set('overlays.particles.opacity', particle_opacity)
cfg.set('overlays.watermark.text', watermark_text)
cfg.set('overlays.watermark.style', watermark_style)
cfg.set('overlays.watermark.position', watermark_position)
cfg.set('overlays.watermark.opacity', watermark_opacity)
cfg.set('overlays.subtitle.style', subtitle_style)
cfg.set('overlays.spectrum.enabled', spectrum_enabled)
cfg.set('overlays.spectrum.style', spectrum_style)
cfg.set('overlays.spectrum.position', spectrum_position)
cfg.set('overlays.spectrum.height', spectrum_height or None)
cfg.set('overlays.spectrum.opacity', spectrum_opacity)
cfg.set('overlays.spectrum.palette', spectrum_palette)
cfg.set('overlays.card.enabled', card_enabled)
cfg.set('overlays.card.title', card_title or None)
cfg.set('overlays.subscribe.enabled', subscribe_enabled)
cfg.set('overlays.subscribe.text', subscribe_text or 'SUBSCRIBE')
cfg.set('overlays.subscribe.appear', subscribe_appear)
cfg.set('output.encoder', 'auto')
cfg.set('output.video_bitrate', video_bitrate or None)
problems = cfg.validate()
print('config OK' if not problems else 'config problems:', problems)

In [ ]:
#@title 6. RUN pipeline (story -> tts -> overlays -> render)
from ytools.pipeline import Pipeline

workdir = '/content/ytools_work/run'
pipe = Pipeline(workdir, cfg, ff=ff)

script_arg = None
if provider == 'manual':
    import os
    SCRIPT = '/content/script.txt'
    if not os.path.isfile(SCRIPT):
        from google.colab import files
        up = files.upload()
        assert up, 'provider=manual wajib upload file script .txt'
        name = list(up.keys())[0]
        open(SCRIPT, 'wb').write(up[name])
    script_arg = SCRIPT

if cfg.get('overlays.watermark.style') == 'logo':
    import os
    LOGO = '/content/logo.png'
    if not os.path.isfile(LOGO):
        from google.colab import files
        up = files.upload()
        assert up, 'watermark style=logo wajib upload file logo (PNG)'
        name = list(up.keys())[0]
        shutil.move(name, LOGO)
    cfg.set('overlays.watermark.logo_path', LOGO)

if cfg.get('overlays.card.enabled'):
    import os
    THUMB = '/content/thumbnail.png'
    if not os.path.isfile(THUMB):
        from google.colab import files
        up = files.upload()
        assert up, 'card enabled wajib upload file thumbnail (PNG/JPG)'
        name = list(up.keys())[0]
        shutil.move(name, THUMB)
    cfg.set('overlays.card.thumbnail_path', THUMB)

art = pipe.run(footage=FOOTAGE, script_path=script_arg)

from IPython.display import HTML
print('duration:', art.duration, 'words:', len(art.words))

In [ ]:
#@title 7. Preview hasil
from IPython.display import HTML, display
from base64 import b64encode

out = art.result.path
mp4 = open(out, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
display(HTML(f'<video width=640 controls><source src="{data_url}"></video>'))
print('size MB:', len(mp4) / 1e6)

In [ ]:
#@title 8. Simpan hasil ke Drive
import shutil, json, datetime

run_dir = os.path.join(DRIVE_ROOT, datetime.date.today().isoformat())
os.makedirs(run_dir, exist_ok=True)

shutil.copy(art.result.path, os.path.join(run_dir, output_name))
shutil.copy(art.script_path, os.path.join(run_dir, output_name.replace('.mp4', '_script.txt')))

meta = {
    'niche': niche,
    'language': language,
    'provider': provider,
    'length_minutes': length_minutes,
    'duration': art.duration,
    'title': art.title,
    'footage': FOOTAGE,
    'created': datetime.datetime.now().isoformat(),
}
meta_path = os.path.join(run_dir, output_name.replace('.mp4', '_meta.json'))
with open(meta_path, 'w', encoding='utf-8') as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)

print('saved to', run_dir)
for f in sorted(os.listdir(run_dir)):
    print('  ', f)

## Catatan

- **Session Colab free** maksimal ~12 jam, idle disconnect ~90 menit. Hasil otomatis tersimpan ke Drive (cell 8) — aman berhenti kapan saja.
- **Reuse**: cell 6 menggunakan cache — rerun cepat kalau footage/particles/watermark sudah ada. Cache key mencakup parameter render (style, density, size, font, dll), jadi ubah parameter itu = render ulang otomatis. **Opacity** (particle, watermark, spectrum) diterapkan saat compositing, jadi ubah opacity = tetap pakai cache tanpa render ulang. **Tombol subscribe**: teks di-cache (PNG), waktu muncul + animasi goyangan di compositing — ubah `subscribe_appear` = no re-render.
- **Ubah parameter cell 5? WAJIB re-run cell 5** sebelum run cell 6, karena nilai form baru dibaca saat cell 5 dieksekusi.
- **Story**: provider `manual` wajib bawa script sendiri. Untuk skrip panjang (>5 menit) gunakan `openai`/`anthropic`.
- **Footage < 30 detik** akan terlalu repetitif; YouTube demote konten loop pendek berulang.
